In [1]:
import os
import glob

BASE_DIR = "./data/1_video_files"

under_100 = 0
exact_100 = 0
over_100 = 0

for ch_name in os.listdir(BASE_DIR):
    ch_dir = os.path.join(BASE_DIR, ch_name)
    if os.path.isdir(ch_dir):
        count = sum(1 for f in os.listdir(ch_dir) if f.lower().endswith(".mp4"))

        if count < 100:
            under_100 += 1
        elif count == 100:
            exact_100 += 1
        else:
            over_100 += 1
            if count > 100:
                print(f"  [초과] {ch_name}: {count}개")

print(f"100개 미만 채널 수: {under_100}")
print(f"100개 채널 수: {exact_100}")
print(f"100개 초과 채널 수: {over_100}")
print(f"총 채널 수: {under_100 + exact_100 + over_100}")

100개 미만 채널 수: 303
100개 채널 수: 664
100개 초과 채널 수: 0
총 채널 수: 967


In [2]:
import os

BASE_DIR = "./data/1_video_files"

exact_100_channels = []
for ch_name in os.listdir(BASE_DIR):
    ch_dir = os.path.join(BASE_DIR, ch_name)
    if not os.path.isdir(ch_dir):
        continue

    count = sum(1 for f in os.listdir(ch_dir) if f.lower().endswith(".mp4"))
    if count == 100:
        exact_100_channels.append(ch_dir)

print("정확히 100개 채널 폴더 수:", len(exact_100_channels))


정확히 100개 채널 폴더 수: 664


In [3]:
import glob

mp4_files = []
for ch_dir in exact_100_channels:
    for f in os.listdir(ch_dir):
        if f.lower().endswith(".mp4"):
            mp4_files.append(os.path.join(ch_dir, f))

print("프레임 추출 대상 mp4 수:", len(mp4_files))



프레임 추출 대상 mp4 수: 66400


In [4]:
import os
import glob
import subprocess
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

INPUT_DIR = "./data/1_video_files"
OUTPUT_DIR = "./data/2_frame_files"
MAX_WORKERS = 32
FPS = 10

def extract_frames(mp4_path: str):
    """영상에서 프레임 추출 (1초당 10장)"""
    rel_path = os.path.relpath(mp4_path, INPUT_DIR)
    channel_name = os.path.dirname(rel_path)
    video_name = os.path.splitext(os.path.basename(rel_path))[0]

    out_dir = os.path.join(OUTPUT_DIR, channel_name, video_name)
    os.makedirs(out_dir, exist_ok=True)

    if len(glob.glob(os.path.join(out_dir, "*.jpg"))) > 0:
        return "skipped", mp4_path, None

    safe_out_dir = out_dir.replace("%", "%%")
    out_pattern = os.path.join(safe_out_dir, "frame_%08d.jpg")

    cmd = [
        "ffmpeg",
        "-i", mp4_path,
        "-threads", "1",
        "-vf", f"fps={FPS}",
        "-hide_banner",
        "-loglevel", "error",
        out_pattern
    ]

    try:
        p = subprocess.run(cmd, capture_output=True, text=True)
        if p.returncode == 0:
            return "success", mp4_path, None
        else:
            return "failed", mp4_path, p.stderr.strip()
    except Exception as e:
        return "error", mp4_path, str(e)

# 하위폴더별 그룹핑
subdir_map = defaultdict(list)
for f in mp4_files:
    subdir = os.path.dirname(f)
    subdir_map[subdir].append(f)

print(f"📁 총 {len(mp4_files)}개 영상, {len(subdir_map)}개 폴더")

stats = {"success": 0, "skipped": 0, "failed": 0, "error": 0}
failed_list = []
error_list = []

for subdir in tqdm(sorted(subdir_map.keys()), desc="폴더 진행"):
    sub_files = subdir_map[subdir]

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(extract_frames, f): f for f in sub_files}

        for fut in as_completed(futures):
            status, path, err = fut.result()
            stats[status] += 1

            if status == "failed":
                failed_list.append(path)
            elif status == "error":
                error_list.append(path)

            if err:
                tqdm.write(f"⚠️ {os.path.basename(path)}: {err}")

retry_files = failed_list + error_list
print("재시도 대상:", len(retry_files))

print(f"\n✅ 완료")
print(f"  - 성공: {stats['success']}")
print(f"  - 스킵 (이미 존재): {stats['skipped']}")
print(f"  - 실패: {stats['failed']}")
print(f"  - 에러: {stats['error']}")

📁 총 66400개 영상, 664개 폴더


폴더 진행:   0%|          | 0/664 [00:00<?, ?it/s]

재시도 대상: 0

✅ 완료
  - 성공: 66400
  - 스킵 (이미 존재): 0
  - 실패: 0
  - 에러: 0
